In [ ]:
import zipfile


with zipfile.ZipFile('satelliteImages.zip', 'r') as zip_ref:
    zip_ref.extractall('satellite')

print("Unzip complete!")

In [ ]:
!ls

In [ ]:
!pip install torch torchvision matplotlib

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
import time

In [ ]:
# Check if GPU is available and move model to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


transform = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

#Load dataset
dataset = ImageFolder(root='satellite/data', transform=transform)

print(dataset.classes)
print(f"Total dataset size: {len(dataset)}")


from torch.utils.data import random_split

train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size]
)

print(f"Train: {train_size}, Val: {val_size}, Test: {test_size}")


batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=2)

#Model
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        self.fc1 = nn.Linear(128 * 30 * 30, 256)
        self.fc2 = nn.Linear(256, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x

model = CNN().to(device)

#Training setup
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

epochs = 7


train_losses = []
val_losses = []

#Training loop
for epoch in range(epochs):
    model.train()
    running_loss = 0
    start_time = time.time()

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=True)

    for batch_idx, (images, labels) in enumerate(progress_bar):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{running_loss/(batch_idx+1):.4f}'
        })


    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    #Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

    val_loss = val_loss / len(val_loader)
    val_losses.append(val_loss)

    scheduler.step()
    epoch_time = time.time() - start_time

    print(f"✅ Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {epoch_time:.2f}s")
    print("-" * 60)

print("Training complete!")

#Plot losses
plt.figure()
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.show()

#Final Test Accuracy
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Evaluating"):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\n📊 Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
import random
import matplotlib.pyplot as plt

#Reverse normalization for displaying images
def imshow(img):
    img = img.cpu().numpy().transpose((1, 2, 0))
    img = np.clip(img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406], 0, 1)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

 #Get class names
class_names = dataset.classes


user_input = input("Enter class name (desert, water, green_area, cloudy): ")

if user_input not in class_names:
    print("❌ Invalid class name!")
else:
    #Find all indices of that class
    indices = [i for i, (_, label) in enumerate(dataset) if class_names[label] == user_input]

    #Pick random image
    idx = random.choice(indices)
    image, label = dataset[idx]

    #Show image
    print(f"\n🖼️ Showing image from class: {user_input}")
    imshow(image)

    #Predict
    model.eval()
    with torch.no_grad():
        img = image.unsqueeze(0).to(device)
        outputs = model(img)
        _, predicted = torch.max(outputs, 1)

    predicted_class = class_names[predicted.item()]

probabilities = torch.nn.functional.softmax(outputs, dim=1)
confidence = probabilities[0][predicted.item()].item()

print(f"🤖 Model prediction: {predicted_class} ({confidence*100:.2f}% confidence)")